#### 流式输出

AI模型边生成边显示响应内容，逐步展示"思考"过程，而非等待完整答案后一次性返回。

#### 模式对比

| 模式 | 特点 |
|:---|:---|
| 普通输出（非流式） | 用户提交问题 → 等待几秒 → 一次性返回完整答案 |
| 流式输出 | 用户提交问题 → AI边生成边显示 → 逐步输出答案 |

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

class AnswerOutput(BaseModel):
    answer: str = Field(description="问题的答案")
    confidence: float = Field(description="答案的置信度")

parser = PydanticOutputParser(pydantic_object=AnswerOutput)

# 定义提示模板
template = """你是一名数学老师，请用{style}风格回答以下问题，并以 JSON 格式返回答案和置信度：
问题：{question}
{format_instructions}"""

prompt = PromptTemplate(
  template=template,
  partial_variables={"format_instructions": parser.get_format_instructions()}
)

llm_chain = prompt | chat 

chunks = []

# 运行链
for chunk in llm_chain.stream({
  'style':'通俗易懂',
  'question':'勾股定理是什么?'
}):
    chunks.append(chunk)
    print(chunk.content,end='',flush=True)

raw_response = "".join([chunk.content for chunk in chunks])
parsed = parser.parse(raw_response)
print("\n\n解析结果：", parsed)
